In [3]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ============================================================
# 설정
# ============================================================

DATA_DIR = Path("../data/raw/cse_cic_ids2018")

# 한 번에 읽을 행 수
# RAM이 부족하면 50_000 정도로 낮춰도 됩니다.
CHUNK_SIZE = 100_000


# ============================================================
# [1] CSV 자동 탐색
# ============================================================

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("=" * 80)
print("[1] CSV 자동 탐색")
print("=" * 80)

print(f"CSV 파일 개수: {len(csv_files)}")

for i, file in enumerate(csv_files, 1):
    print(f"{i:2d}. {file.name}")

if not csv_files:
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다: {DATA_DIR.resolve()}"
    )


# ============================================================
# 결과 저장용
# ============================================================

file_results = []

column_sets = {}
dtype_results = {}
label_results = {}

total_label_counts = Counter()

reference_columns = None


# ============================================================
# 파일별 검사
# ============================================================

for file_index, file in enumerate(csv_files, 1):

    print("\n")
    print("#" * 80)
    print(f"[{file_index}/{len(csv_files)}] {file.name}")
    print("#" * 80)

    # --------------------------------------------------------
    # [2] 파일 크기
    # --------------------------------------------------------

    file_size_bytes = file.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)
    file_size_gb = file_size_bytes / (1024 ** 3)

    print("\n[2] 파일 크기")

    if file_size_gb >= 1:
        print(f"{file_size_gb:.2f} GB")
    else:
        print(f"{file_size_mb:.2f} MB")


    # --------------------------------------------------------
    # Header만 읽어서 컬럼 확인
    # --------------------------------------------------------

    header = pd.read_csv(file, nrows=0)

    # 혹시 컬럼명 앞뒤에 공백이 있는 경우 대비
    columns = header.columns.str.strip().tolist()

    column_sets[file.name] = columns

    print("\n컬럼 수:", len(columns))


    # --------------------------------------------------------
    # [4] 컬럼 일치 여부
    # --------------------------------------------------------

    print("\n[4] 컬럼 일치 여부")

    if reference_columns is None:

        reference_columns = columns

        print("기준 컬럼으로 설정")

    else:

        if columns == reference_columns:

            print("OK - 기준 파일과 컬럼 동일")

        else:

            print("WARNING - 컬럼 불일치")

            missing = set(reference_columns) - set(columns)
            extra = set(columns) - set(reference_columns)

            print("누락 컬럼:", missing)
            print("추가 컬럼:", extra)


    # --------------------------------------------------------
    # 파일 통계 변수
    # --------------------------------------------------------

    row_count = 0

    null_count = Counter()

    inf_count = Counter()

    label_counter = Counter()

    memory_usage = 0

    duplicate_count = 0

    dtype_info = None


    # --------------------------------------------------------
    # Chunk 단위 읽기
    # --------------------------------------------------------

    print("\n데이터 검사 중...")

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            file,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        1
    ):

        # 컬럼명 정리
        chunk.columns = chunk.columns.str.strip()

        # ----------------------------------------------------
        # [3] 행 개수
        # ----------------------------------------------------

        row_count += len(chunk)


        # ----------------------------------------------------
        # [5] dtype
        # ----------------------------------------------------

        if dtype_info is None:

            dtype_info = chunk.dtypes.astype(str).to_dict()


        # ----------------------------------------------------
        # [6] NaN
        # ----------------------------------------------------

        chunk_nulls = chunk.isna().sum()

        for column, count in chunk_nulls.items():

            if count > 0:
                null_count[column] += int(count)


        # ----------------------------------------------------
        # [7] Inf / -Inf
        # ----------------------------------------------------

        numeric_columns = chunk.select_dtypes(
            include=[np.number]
        ).columns

        if len(numeric_columns) > 0:

            inf_mask = np.isinf(
                chunk[numeric_columns]
            )

            chunk_inf = inf_mask.sum()

            for column, count in chunk_inf.items():

                if count > 0:
                    inf_count[column] += int(count)


        # ----------------------------------------------------
        # [8] Duplicate
        # ----------------------------------------------------

        duplicate_count += int(
            chunk.duplicated().sum()
        )


        # ----------------------------------------------------
        # [9] Label 분포
        # ----------------------------------------------------

        if "Label" in chunk.columns:

            labels = (
                chunk["Label"]
                .astype(str)
                .str.strip()
                .value_counts()
            )

            label_counter.update(
                labels.to_dict()
            )

            total_label_counts.update(
                labels.to_dict()
            )


        # ----------------------------------------------------
        # [11] 메모리 사용량 추정
        # ----------------------------------------------------

        memory_usage += chunk.memory_usage(
            deep=True
        ).sum()


        # 진행상황
        if chunk_number % 10 == 0:

            print(
                f"  chunk={chunk_number:,} "
                f"rows={row_count:,}"
            )


    # ========================================================
    # 파일 검사 완료
    # ========================================================

    print("\n" + "-" * 80)

    # --------------------------------------------------------
    # [3] 행 / 열
    # --------------------------------------------------------

    print("\n[3] 행 / 열 개수")

    print(f"Rows    : {row_count:,}")
    print(f"Columns : {len(columns):,}")


    # --------------------------------------------------------
    # [5] dtype
    # --------------------------------------------------------

    print("\n[5] dtype")

    dtype_series = pd.Series(dtype_info)

    print(dtype_series.value_counts())

    dtype_results[file.name] = dtype_info


    # --------------------------------------------------------
    # [6] NaN
    # --------------------------------------------------------

    print("\n[6] NaN")

    total_null = sum(null_count.values())

    print(f"총 NaN: {total_null:,}")

    if null_count:

        print(
            pd.Series(null_count)
            .sort_values(ascending=False)
        )

    else:

        print("NaN 없음")


    # --------------------------------------------------------
    # [7] Inf
    # --------------------------------------------------------

    print("\n[7] Inf / -Inf")

    total_inf = sum(inf_count.values())

    print(f"총 Inf: {total_inf:,}")

    if inf_count:

        print(
            pd.Series(inf_count)
            .sort_values(ascending=False)
        )

    else:

        print("Inf 없음")


    # --------------------------------------------------------
    # [8] Duplicate
    # --------------------------------------------------------

    print("\n[8] Duplicate")

    print(
        f"중복 행: {duplicate_count:,}"
    )

    print(
        f"중복 비율: "
        f"{duplicate_count / row_count * 100:.4f}%"
        if row_count
        else "0%"
    )


    # --------------------------------------------------------
    # [9] Label 분포
    # --------------------------------------------------------

    print("\n[9] Label 분포")

    label_series = pd.Series(label_counter).sort_values(
        ascending=False
    )

    print(label_series)

    label_results[file.name] = dict(label_counter)


    # --------------------------------------------------------
    # [11] 메모리 사용량
    # --------------------------------------------------------

    memory_mb = memory_usage / (1024 ** 2)
    memory_gb = memory_usage / (1024 ** 3)

    print("\n[11] DataFrame 예상 메모리")

    if memory_gb >= 1:
        print(f"{memory_gb:.2f} GB")
    else:
        print(f"{memory_mb:.2f} MB")


    # --------------------------------------------------------
    # 결과 저장
    # --------------------------------------------------------

    file_results.append(
        {
            "file": file.name,
            "file_size_mb": round(file_size_mb, 2),
            "rows": row_count,
            "columns": len(columns),
            "nan_count": total_null,
            "inf_count": total_inf,
            "duplicate_count": duplicate_count,
            "memory_mb": round(memory_mb, 2),
        }
    )


# ============================================================
# 전체 결과
# ============================================================

print("\n\n")
print("=" * 80)
print("전체 데이터셋 검사 결과")
print("=" * 80)


# ============================================================
# 파일 요약
# ============================================================

summary_df = pd.DataFrame(file_results)

print("\n파일별 요약")
print(summary_df.to_string(index=False))


# ============================================================
# [4] 전체 컬럼 일치 여부
# ============================================================

print("\n")
print("=" * 80)
print("[4] 전체 CSV 컬럼 일치 여부")
print("=" * 80)

all_columns_match = all(
    columns == reference_columns
    for columns in column_sets.values()
)

print("전체 컬럼 동일:", all_columns_match)

if not all_columns_match:

    for filename, columns in column_sets.items():

        if columns != reference_columns:

            print(f"\n{filename}")

            print(
                "누락:",
                set(reference_columns) - set(columns)
            )

            print(
                "추가:",
                set(columns) - set(reference_columns)
            )


# ============================================================
# [10] 전체 Label 분포
# ============================================================

print("\n")
print("=" * 80)
print("[10] 전체 Label 분포")
print("=" * 80)

total_labels = pd.Series(
    total_label_counts
).sort_values(
    ascending=False
)

print(total_labels)

print("\n전체 Label 비율 (%)")

label_ratio = (
    total_labels
    / total_labels.sum()
    * 100
)

print(label_ratio.round(4))


# ============================================================
# [11] 전체 메모리 사용량 추정
# ============================================================

print("\n")
print("=" * 80)
print("[11] 전체 메모리 사용량 추정")
print("=" * 80)

total_file_size = summary_df["file_size_mb"].sum()

total_memory = summary_df["memory_mb"].sum()

print(
    f"CSV 실제 용량 : "
    f"{total_file_size / 1024:.2f} GB"
)

print(
    f"Pandas 예상 메모리 : "
    f"{total_memory / 1024:.2f} GB"
)

print(
    f"총 행 개수 : "
    f"{summary_df['rows'].sum():,}"
)


# ============================================================
# 결과 CSV 저장
# ============================================================

RESULT_DIR = Path("results/dataset_inspection")

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

summary_df.to_csv(
    RESULT_DIR / "file_summary.csv",
    index=False
)

total_labels.rename(
    "count"
).to_csv(
    RESULT_DIR / "label_distribution.csv"
)

label_ratio.rename(
    "percentage"
).to_csv(
    RESULT_DIR / "label_distribution_ratio.csv"
)

print("\n결과 저장 완료:")
print(RESULT_DIR.resolve())

[1] CSV 자동 탐색
CSV 파일 개수: 10
 1. Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
 2. Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
 3. Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
 4. Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
 5. Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
 6. Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
 7. Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
 8. Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
 9. Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
10. Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


################################################################################
[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
################################################################################

[2] 파일 크기
336.04 MB

컬럼 수: 80

[4] 컬럼 일치 여부
기준 컬럼으로 설정

데이터 검사 중...
  chunk=10 rows=1,000,000

--------------------------------------------------------------------------------

[3] 행 / 열 개수
Rows    : 1,048,575
Colum

In [4]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/cse_cic_ids2018")

csv_files = sorted(DATA_DIR.glob("*.csv"))

schemas = {}

for file in csv_files:
    header = pd.read_csv(file, nrows=0)

    # 컬럼명 앞뒤 공백 제거
    columns = [col.strip() for col in header.columns]

    schemas[file.name] = columns

    print("=" * 100)
    print(file.name)
    print(f"컬럼 수: {len(columns)}")

    for i, col in enumerate(columns, 1):
        print(f"{i:2d}. {col}")

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
컬럼 수: 80
 1. Dst Port
 2. Protocol
 3. Timestamp
 4. Flow Duration
 5. Tot Fwd Pkts
 6. Tot Bwd Pkts
 7. TotLen Fwd Pkts
 8. TotLen Bwd Pkts
 9. Fwd Pkt Len Max
10. Fwd Pkt Len Min
11. Fwd Pkt Len Mean
12. Fwd Pkt Len Std
13. Bwd Pkt Len Max
14. Bwd Pkt Len Min
15. Bwd Pkt Len Mean
16. Bwd Pkt Len Std
17. Flow Byts/s
18. Flow Pkts/s
19. Flow IAT Mean
20. Flow IAT Std
21. Flow IAT Max
22. Flow IAT Min
23. Fwd IAT Tot
24. Fwd IAT Mean
25. Fwd IAT Std
26. Fwd IAT Max
27. Fwd IAT Min
28. Bwd IAT Tot
29. Bwd IAT Mean
30. Bwd IAT Std
31. Bwd IAT Max
32. Bwd IAT Min
33. Fwd PSH Flags
34. Bwd PSH Flags
35. Fwd URG Flags
36. Bwd URG Flags
37. Fwd Header Len
38. Bwd Header Len
39. Fwd Pkts/s
40. Bwd Pkts/s
41. Pkt Len Min
42. Pkt Len Max
43. Pkt Len Mean
44. Pkt Len Std
45. Pkt Len Var
46. FIN Flag Cnt
47. SYN Flag Cnt
48. RST Flag Cnt
49. PSH Flag Cnt
50. ACK Flag Cnt
51. URG Flag Cnt
52. CWE Flag Count
53. ECE Flag Cnt
54. Down/Up Ratio
55. Pkt S

In [5]:
all_sets = {
    filename: set(columns)
    for filename, columns in schemas.items()
}

common_columns = set.intersection(*all_sets.values())
union_columns = set.union(*all_sets.values())

print("\n공통 컬럼 수:", len(common_columns))
print("전체 컬럼 수:", len(union_columns))

print("\n=== 모든 파일에 공통으로 존재하는 컬럼 ===")
for col in sorted(common_columns):
    print(col)

print("\n=== 일부 파일에만 존재하는 컬럼 ===")

for filename, columns in all_sets.items():

    extra = columns - common_columns

    if extra:
        print(f"\n{filename}")
        for col in sorted(extra):
            print(" +", col)


공통 컬럼 수: 80
전체 컬럼 수: 84

=== 모든 파일에 공통으로 존재하는 컬럼 ===
ACK Flag Cnt
Active Max
Active Mean
Active Min
Active Std
Bwd Blk Rate Avg
Bwd Byts/b Avg
Bwd Header Len
Bwd IAT Max
Bwd IAT Mean
Bwd IAT Min
Bwd IAT Std
Bwd IAT Tot
Bwd PSH Flags
Bwd Pkt Len Max
Bwd Pkt Len Mean
Bwd Pkt Len Min
Bwd Pkt Len Std
Bwd Pkts/b Avg
Bwd Pkts/s
Bwd Seg Size Avg
Bwd URG Flags
CWE Flag Count
Down/Up Ratio
Dst Port
ECE Flag Cnt
FIN Flag Cnt
Flow Byts/s
Flow Duration
Flow IAT Max
Flow IAT Mean
Flow IAT Min
Flow IAT Std
Flow Pkts/s
Fwd Act Data Pkts
Fwd Blk Rate Avg
Fwd Byts/b Avg
Fwd Header Len
Fwd IAT Max
Fwd IAT Mean
Fwd IAT Min
Fwd IAT Std
Fwd IAT Tot
Fwd PSH Flags
Fwd Pkt Len Max
Fwd Pkt Len Mean
Fwd Pkt Len Min
Fwd Pkt Len Std
Fwd Pkts/b Avg
Fwd Pkts/s
Fwd Seg Size Avg
Fwd Seg Size Min
Fwd URG Flags
Idle Max
Idle Mean
Idle Min
Idle Std
Init Bwd Win Byts
Init Fwd Win Byts
Label
PSH Flag Cnt
Pkt Len Max
Pkt Len Mean
Pkt Len Min
Pkt Len Std
Pkt Len Var
Pkt Size Avg
Protocol
RST Flag Cnt
SYN Flag Cnt
Subflow B

In [6]:
from pathlib import Path

import pandas as pd


# ============================================================
# 경로 설정
# ============================================================

DATA_DIR = Path("../data/raw/cse_cic_ids2018")
OUTPUT_DIR = Path("../data/processed/cse_cic_ids2018")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "cse_cic_ids2018_merged.csv"

CHUNK_SIZE = 100_000


# ============================================================
# CSV 탐색
# ============================================================

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"CSV 파일: {len(csv_files)}개")

if not csv_files:
    raise FileNotFoundError(DATA_DIR)


# ============================================================
# 모든 파일의 컬럼 확인
# ============================================================

schemas = {}

for file in csv_files:

    header = pd.read_csv(file, nrows=0)

    columns = [
        column.strip()
        for column in header.columns
    ]

    schemas[file.name] = columns


# ============================================================
# 공통 컬럼 계산
# ============================================================

column_sets = [
    set(columns)
    for columns in schemas.values()
]

common_set = set.intersection(*column_sets)


# 첫 번째 파일의 컬럼 순서를 유지
first_columns = next(iter(schemas.values()))

common_columns = [
    column
    for column in first_columns
    if column in common_set
]


print(f"공통 컬럼: {len(common_columns)}개")


# ============================================================
# 제외되는 컬럼 확인
# ============================================================

print("\n=== 공통 스키마에서 제외되는 컬럼 ===")

for filename, columns in schemas.items():

    excluded = [
        column
        for column in columns
        if column not in common_set
    ]

    if excluded:

        print(f"\n{filename}")

        for column in excluded:
            print(" -", column)


# ============================================================
# 기존 병합 파일 삭제
# ============================================================

if OUTPUT_FILE.exists():

    print("\n기존 병합 파일 삭제")
    OUTPUT_FILE.unlink()


# ============================================================
# Chunk 단위 병합
# ============================================================

first_write = True
total_rows = 0
removed_header_rows = 0


for file_index, file in enumerate(csv_files, 1):

    print("\n" + "=" * 80)

    print(
        f"[{file_index}/{len(csv_files)}] "
        f"{file.name}"
    )

    print("=" * 80)

    file_rows = 0


    for chunk_index, chunk in enumerate(

        pd.read_csv(
            file,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),

        1
    ):

        # ----------------------------------------------------
        # 컬럼명 공백 제거
        # ----------------------------------------------------

        chunk.columns = chunk.columns.str.strip()


        # ----------------------------------------------------
        # 공통 80개 컬럼만 선택
        # ----------------------------------------------------

        chunk = chunk[common_columns].copy()


        # ----------------------------------------------------
        # Label 정리
        # ----------------------------------------------------

        if "Label" in chunk.columns:

            chunk["Label"] = (
                chunk["Label"]
                .astype(str)
                .str.strip()
            )


            # 잘못 들어간 Header 행 제거

            invalid_header = (
                chunk["Label"] == "Label"
            )

            removed_header_rows += int(
                invalid_header.sum()
            )

            chunk = chunk[
                ~invalid_header
            ]


        # ----------------------------------------------------
        # 원본 파일 추적 컬럼
        # ----------------------------------------------------

        chunk["source_file"] = file.name


        # ----------------------------------------------------
        # 저장
        # ----------------------------------------------------

        chunk.to_csv(
            OUTPUT_FILE,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        first_write = False

        file_rows += len(chunk)
        total_rows += len(chunk)


        print(
            f"chunk={chunk_index:3d} | "
            f"file_rows={file_rows:,} | "
            f"total={total_rows:,}"
        )


# ============================================================
# 완료
# ============================================================

print("\n" + "=" * 80)
print("병합 완료")
print("=" * 80)

print(f"저장 위치 : {OUTPUT_FILE}")
print(f"총 행 수  : {total_rows:,}")
print(f"컬럼 수   : {len(common_columns) + 1}")
print(f"컬럼 목록 : {common_columns + ['source_file']}")
print(f"제거된 잘못된 Header 행 : {removed_header_rows:,}")

CSV 파일: 10개
공통 컬럼: 80개

=== 공통 스키마에서 제외되는 컬럼 ===

Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
 - Flow ID
 - Src IP
 - Src Port
 - Dst IP

[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
chunk=  1 | file_rows=100,000 | total=100,000
chunk=  2 | file_rows=200,000 | total=200,000
chunk=  3 | file_rows=300,000 | total=300,000
chunk=  4 | file_rows=400,000 | total=400,000
chunk=  5 | file_rows=500,000 | total=500,000
chunk=  6 | file_rows=600,000 | total=600,000
chunk=  7 | file_rows=700,000 | total=700,000
chunk=  8 | file_rows=800,000 | total=800,000
chunk=  9 | file_rows=900,000 | total=900,000
chunk= 10 | file_rows=1,000,000 | total=1,000,000
chunk= 11 | file_rows=1,048,575 | total=1,048,575

[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
chunk=  1 | file_rows=100,000 | total=1,148,575
chunk=  2 | file_rows=200,000 | total=1,248,575
chunk=  3 | file_rows=300,000 | total=1,348,575
chunk=  4 | file_rows=400,000 | total=1,448,575
chunk=  5 | file_rows=500,000 | total=1

In [7]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ============================================================
# 0. 설정
# ============================================================

DATA_FILE = Path(
    "../data/processed/cse_cic_ids2018/cse_cic_ids2018_merged.csv"
)

RESULT_DIR = Path("../results/data_quality")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

# Feature 분포 / 상관관계 분석용 최대 샘플
SAMPLE_SIZE = 300_000

# 재현성
RANDOM_STATE = 42

# Target / Metadata
TARGET_COLUMN = "Label"
METADATA_COLUMNS = ["source_file", "Timestamp"]


if not DATA_FILE.exists():
    raise FileNotFoundError(DATA_FILE)


print("=" * 80)
print("CSE-CIC-IDS2018 Data Quality Analysis")
print("=" * 80)
print("Input :", DATA_FILE)
print("Output:", RESULT_DIR)


# ============================================================
# 1. Header / Feature 확인
# ============================================================

header = pd.read_csv(DATA_FILE, nrows=0)

columns = header.columns.str.strip().tolist()

print("\n전체 컬럼 수:", len(columns))
print(columns)


# ============================================================
# 2. 전체 데이터 품질 검사
# ============================================================

total_rows = 0
total_memory = 0

null_counts = Counter()
inf_counts = Counter()
label_counts = Counter()

dtype_sets = {
    column: set()
    for column in columns
}

source_counts = Counter()


# ============================================================
# Duplicate 검사
#
# 주의:
# chunk.duplicated()만 사용하면 chunk 경계 중복을 놓친다.
#
# 1,600만 행 전체 중복을 완전히 메모리에 저장하는 것도
# 비용이 크므로 pandas hash를 사용한다.
# ============================================================

seen_hashes = set()
duplicate_count = 0


# ============================================================
# 분석용 Reservoir Sample
# ============================================================

sample_parts = []
sample_rows_collected = 0


print("\n전체 데이터 검사 시작...")


for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    1
):

    chunk.columns = chunk.columns.str.strip()

    rows = len(chunk)

    total_rows += rows


    # --------------------------------------------------------
    # 데이터 타입
    # --------------------------------------------------------

    for column in chunk.columns:

        dtype_sets[column].add(
            str(chunk[column].dtype)
        )


    # --------------------------------------------------------
    # 결측치
    # --------------------------------------------------------

    nulls = chunk.isna().sum()

    for column, count in nulls.items():

        if count:
            null_counts[column] += int(count)


    # --------------------------------------------------------
    # Inf / -Inf
    # --------------------------------------------------------

    numeric_columns = chunk.select_dtypes(
        include=[np.number]
    ).columns

    if len(numeric_columns):

        values = chunk[numeric_columns]

        inf_result = np.isinf(values).sum()

        for column, count in inf_result.items():

            if count:
                inf_counts[column] += int(count)


    # --------------------------------------------------------
    # Label
    # --------------------------------------------------------

    if TARGET_COLUMN in chunk.columns:

        labels = (
            chunk[TARGET_COLUMN]
            .astype(str)
            .str.strip()
            .value_counts()
        )

        label_counts.update(
            labels.to_dict()
        )


    # --------------------------------------------------------
    # source_file
    # --------------------------------------------------------

    if "source_file" in chunk.columns:

        sources = (
            chunk["source_file"]
            .astype(str)
            .value_counts()
        )

        source_counts.update(
            sources.to_dict()
        )


    # --------------------------------------------------------
    # Duplicate
    # --------------------------------------------------------
    #
    # source_file은 우리가 추가한 provenance 정보이므로
    # 실제 Flow 중복 검사에서는 제외한다.
    #
    # Timestamp는 원본 Feature이므로 유지한다.
    # --------------------------------------------------------

    duplicate_columns = [
        column
        for column in chunk.columns
        if column != "source_file"
    ]

    hashes = pd.util.hash_pandas_object(
        chunk[duplicate_columns],
        index=False
    ).to_numpy()


    for value in hashes:

        value = int(value)

        if value in seen_hashes:
            duplicate_count += 1

        else:
            seen_hashes.add(value)


    # --------------------------------------------------------
    # 메모리 사용량
    # --------------------------------------------------------

    total_memory += int(
        chunk.memory_usage(
            deep=True
        ).sum()
    )


    # --------------------------------------------------------
    # 분석용 샘플
    # --------------------------------------------------------

    # 각 chunk에서 일부를 랜덤 추출한다.
    # 최종적으로 SAMPLE_SIZE 근처로 제한한다.

    if sample_rows_collected < SAMPLE_SIZE:

        remaining = (
            SAMPLE_SIZE
            - sample_rows_collected
        )

        take = min(
            max(1, int(rows * 0.02)),
            remaining
        )

        if take > 0:

            sampled = chunk.sample(
                n=min(take, rows),
                random_state=(
                    RANDOM_STATE
                    + chunk_number
                )
            )

            sample_parts.append(sampled)

            sample_rows_collected += len(sampled)


    print(
        f"chunk={chunk_number:3d} | "
        f"rows={total_rows:,} | "
        f"duplicates={duplicate_count:,}"
    )


# ============================================================
# Sample 통합
# ============================================================

sample_df = pd.concat(
    sample_parts,
    ignore_index=True
)

if len(sample_df) > SAMPLE_SIZE:

    sample_df = sample_df.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )


print("\n전체 검사 완료")
print("총 행:", f"{total_rows:,}")
print("분석 Sample:", f"{len(sample_df):,}")


# ============================================================
# 3. dataset_profile.csv
# ============================================================

file_size_bytes = DATA_FILE.stat().st_size

file_size_gb = (
    file_size_bytes
    / 1024 ** 3
)

memory_gb = (
    total_memory
    / 1024 ** 3
)


profile = {
    "dataset": "CSE-CIC-IDS2018",
    "rows": total_rows,
    "columns": len(columns),

    "feature_columns": len([
        column
        for column in columns
        if column not in [
            TARGET_COLUMN,
            "source_file"
        ]
    ]),

    "file_size_gb": round(
        file_size_gb,
        3
    ),

    "estimated_pandas_memory_gb": round(
        memory_gb,
        3
    ),

    "total_nan": sum(
        null_counts.values()
    ),

    "total_inf": sum(
        inf_counts.values()
    ),

    "duplicate_rows": duplicate_count,

    "duplicate_ratio": (
        duplicate_count
        / total_rows
        if total_rows
        else 0
    ),

    "class_count": len(
        label_counts
    ),

    "sample_size_for_distribution": len(
        sample_df
    ),
}


dataset_profile_df = pd.DataFrame(
    [profile]
)

dataset_profile_df.to_csv(
    RESULT_DIR / "dataset_profile.csv",
    index=False
)


# ============================================================
# 4. class_distribution.csv
# ============================================================

class_distribution = pd.DataFrame(
    label_counts.items(),
    columns=[
        "Label",
        "count"
    ]
)

class_distribution = (
    class_distribution
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(drop=True)
)


class_distribution["percentage"] = (
    class_distribution["count"]
    / total_rows
    * 100
)


class_distribution["ratio_to_majority"] = (
    class_distribution["count"]
    / class_distribution["count"].max()
)


class_distribution.to_csv(
    RESULT_DIR / "class_distribution.csv",
    index=False
)


# ============================================================
# 5. data_quality_report.csv
# ============================================================

quality_rows = []


for column in columns:

    dtype_value = "|".join(
        sorted(dtype_sets[column])
    )

    nan_count = int(
        null_counts.get(
            column,
            0
        )
    )

    inf_count = int(
        inf_counts.get(
            column,
            0
        )
    )

    quality_rows.append(
        {
            "column": column,

            "dtype": dtype_value,

            "nan_count": nan_count,

            "nan_percentage": (
                nan_count
                / total_rows
                * 100
            ),

            "inf_count": inf_count,

            "inf_percentage": (
                inf_count
                / total_rows
                * 100
            ),
        }
    )


data_quality_report = pd.DataFrame(
    quality_rows
)


# ============================================================
# 6. 이상값 분석
#
# IQR 기준
#
# 중요한 점:
# 여기서는 이상값을 삭제하지 않는다.
# 공격 데이터 자체가 극단값일 가능성이 있기 때문이다.
# ============================================================

exclude_columns = {
    TARGET_COLUMN,
    "source_file",
    "Timestamp",
}


numeric_sample = (
    sample_df
    .drop(
        columns=[
            column
            for column in exclude_columns
            if column in sample_df.columns
        ],
        errors="ignore"
    )
    .select_dtypes(
        include=[np.number]
    )
)


# Inf는 quantile 계산에서 방해되므로
# 분석 시에만 NaN으로 바꾼다.

numeric_clean = numeric_sample.replace(
    [np.inf, -np.inf],
    np.nan
)


outlier_results = []


for column in numeric_clean.columns:

    series = numeric_clean[column].dropna()

    if series.empty:
        continue


    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr


    # IQR=0인 Feature도 기록한다.
    if iqr == 0:

        outlier_count = int(
            (
                (series < lower_bound)
                |
                (series > upper_bound)
            ).sum()
        )

    else:

        outlier_count = int(
            (
                (series < lower_bound)
                |
                (series > upper_bound)
            ).sum()
        )


    outlier_results.append(
        {
            "feature": column,

            "sample_count": len(series),

            "min": series.min(),

            "q1": q1,

            "median": series.median(),

            "q3": q3,

            "max": series.max(),

            "iqr": iqr,

            "lower_bound": lower_bound,

            "upper_bound": upper_bound,

            "outlier_count": outlier_count,

            "outlier_percentage": (
                outlier_count
                / len(series)
                * 100
            ),
        }
    )


outlier_df = pd.DataFrame(
    outlier_results
)

outlier_df = outlier_df.sort_values(
    "outlier_percentage",
    ascending=False
)

outlier_df.to_csv(
    RESULT_DIR / "outlier_report.csv",
    index=False
)


# ============================================================
# 7. Feature 분포
# ============================================================

distribution_rows = []


for column in numeric_clean.columns:

    series = numeric_clean[column].dropna()

    if series.empty:
        continue


    distribution_rows.append(
        {
            "feature": column,

            "count": len(series),

            "mean": series.mean(),

            "std": series.std(),

            "min": series.min(),

            "p01": series.quantile(0.01),

            "p05": series.quantile(0.05),

            "p25": series.quantile(0.25),

            "median": series.median(),

            "p75": series.quantile(0.75),

            "p95": series.quantile(0.95),

            "p99": series.quantile(0.99),

            "max": series.max(),

            "zero_count": int(
                (series == 0).sum()
            ),

            "zero_percentage": (
                (series == 0).mean()
                * 100
            ),

            "unique_count": int(
                series.nunique()
            ),
        }
    )


feature_distribution = pd.DataFrame(
    distribution_rows
)

feature_distribution.to_csv(
    RESULT_DIR / "feature_distribution.csv",
    index=False
)


# ============================================================
# 8. Feature 상관관계
# ============================================================

correlation_matrix = numeric_clean.corr(
    method="pearson"
)

correlation_matrix.to_csv(
    RESULT_DIR / "feature_correlation_matrix.csv"
)


# ============================================================
# 높은 상관관계 Feature Pair 추출
# ============================================================

correlation_pairs = []


correlation_columns = (
    correlation_matrix.columns
)


for i in range(
    len(correlation_columns)
):

    for j in range(
        i + 1,
        len(correlation_columns)
    ):

        feature_a = (
            correlation_columns[i]
        )

        feature_b = (
            correlation_columns[j]
        )

        correlation = (
            correlation_matrix.loc[
                feature_a,
                feature_b
            ]
        )

        if pd.isna(correlation):
            continue


        correlation_pairs.append(
            {
                "feature_a": feature_a,
                "feature_b": feature_b,
                "correlation": correlation,
                "abs_correlation": abs(
                    correlation
                ),
            }
        )


correlation_pairs_df = pd.DataFrame(
    correlation_pairs
)

correlation_pairs_df = (
    correlation_pairs_df
    .sort_values(
        "abs_correlation",
        ascending=False
    )
)


correlation_pairs_df.to_csv(
    RESULT_DIR
    / "feature_correlation_pairs.csv",
    index=False
)


# 강한 상관관계만 별도 저장

high_correlation = (
    correlation_pairs_df[
        correlation_pairs_df[
            "abs_correlation"
        ] >= 0.90
    ]
)


high_correlation.to_csv(
    RESULT_DIR
    / "high_correlation_features.csv",
    index=False
)


# ============================================================
# 9. 날짜(source_file) × Label
# ============================================================

if (
    "source_file" in sample_df.columns
    and TARGET_COLUMN in sample_df.columns
):

    source_label = pd.crosstab(
        sample_df["source_file"],
        sample_df[TARGET_COLUMN]
    )

    source_label.to_csv(
        RESULT_DIR
        / "source_file_label_distribution_sample.csv"
    )


# ============================================================
# 10. 분석 Sample 저장
# ============================================================

# CSV보다 이후 분석이 빠른 Parquet 사용

sample_df.to_parquet(
    RESULT_DIR
    / "eda_sample.parquet",
    index=False
)


# ============================================================
# 11. 최종 출력
# ============================================================

print("\n")
print("=" * 80)
print("DATA QUALITY ANALYSIS COMPLETE")
print("=" * 80)


print("\n[Dataset Profile]")
print(
    dataset_profile_df.to_string(
        index=False
    )
)


print("\n[Class Distribution]")
print(
    class_distribution.to_string(
        index=False
    )
)


print("\n[Top 20 Outlier Features]")

print(
    outlier_df[
        [
            "feature",
            "outlier_count",
            "outlier_percentage",
        ]
    ]
    .head(20)
    .to_string(index=False)
)


print("\n[Top 20 Correlated Feature Pairs]")

print(
    correlation_pairs_df[
        [
            "feature_a",
            "feature_b",
            "correlation",
        ]
    ]
    .head(20)
    .to_string(index=False)
)


print("\n결과 저장 위치:")
print(RESULT_DIR.resolve())

CSE-CIC-IDS2018 Data Quality Analysis
Input : ../data/processed/cse_cic_ids2018/cse_cic_ids2018_merged.csv
Output: ../results/data_quality

전체 컬럼 수: 81
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 

## 데이터 품질 분석

CSE-CIC-IDS2018 데이터를 모델 학습에 사용하기 전 데이터 구조와 품질을 점검하였다.

### 분석 결과

- 전체 데이터: **16,232,943건**
- 전체 컬럼: **81개**
  - ML Feature: **79개**
  - Target: `Label`
  - 메타데이터: `source_file`
- CSV 크기: 약 **7.03 GB**
- 메모리 사용량 추정: 약 **10.94 GB**
- 결측치(NaN): **59,721건**
- ±Inf: **131,799건**
- 중복 데이터: **422,815건 (약 2.60%)**
- 공격 클래스: **15개**

### 클래스 불균형

`Benign` 데이터가 **83.07%**로 대부분을 차지했으며, 가장 적은 `SQL Injection`은 **87건(0.00054%)**에 불과했다.

따라서 심각한 클래스 불균형이 존재하므로 모델 평가 시 Accuracy만 사용하지 않고 **Precision, Recall, F1-score, Macro F1, Confusion Matrix** 등을 함께 사용한다.

### Feature 품질

IAT 관련 Feature에서 IQR 기준 이상값이 다수 발견되었다. 그러나 네트워크 공격의 특성상 극단값 자체가 공격 행동을 나타낼 가능성이 있으므로 이상값을 일괄 제거하지 않고 Label별 분포를 추가 분석한다.

또한 일부 Feature 간 매우 높은 상관관계가 확인되었다. 예를 들어 `TotLen Fwd Pkts`와 `Subflow Fwd Byts`, `Tot Fwd Pkts`와 `Subflow Fwd Pkts` 등은 상관계수 1.0으로 나타났다. 이후 Feature Selection 단계에서 중복 정보 여부를 검토한다.

### 산출물

- `dataset_profile.csv`
- `class_distribution.csv`
- `outlier_report.csv`
- `feature_distribution.csv`
- `feature_correlation_matrix.csv`
- `high_correlation_features.csv`

1. 기본 Flow 정보
컬럼	의미	분석 포인트
Dst Port	목적지 포트	어떤 서비스로 통신하는지
Protocol	IP 프로토콜 번호	TCP/UDP 등 프로토콜별 공격 차이
Timestamp	Flow 발생 시각	시간대·시간순 공격 분석
Flow Duration	Flow 지속시간	짧은 반복 연결/장시간 연결 분석
source_file	원본 CSV	날짜/시나리오 추적용
Label	정상/공격 클래스	ML Target
여기서 source_file은 우리가 추가한 메타데이터이므로 ML Feature가 아니다.
Timestamp역시 그대로 XGBoost에 넣기보다는 EDA 또는 hour, day같은 파생변수를 만드는 용도로 우선 사용하는 게 좋아.

2. 패킷 수와 데이터 전송량
네트워크가 얼마나 많은 데이터를 주고받았는가를 나타낸다.
컬럼	의미
Tot Fwd Pkts	순방향 전체 패킷 수
Tot Bwd Pkts	역방향 전체 패킷 수
TotLen Fwd Pkts	순방향 패킷 전체 길이
TotLen Bwd Pkts	역방향 패킷 전체 길이
Flow Byts/s	초당 전송 Byte
Flow Pkts/s	초당 전체 Packet
Fwd Pkts/s	초당 순방향 Packet
Bwd Pkts/s	초당 역방향 Packet
Down/Up Ratio	다운로드/업로드 방향 비율
특히:
Flow Pkts/s ↑ Flow Byts/s ↑
같은 현상은 대량의 트래픽이 발생하는 행동을 조사할 때 중요한 분석 대상이 될 수 있어.
다만 값이 높다고 바로 DDoS라고 판단하면 안 되고, 실제 Label별 분포를 비교해서 확인해야 한다.

3. 순방향 패킷 크기
Fwd는 출발지 → 목적지 방향이다.
컬럼	의미
Fwd Pkt Len Max	순방향 패킷 최대 길이
Fwd Pkt Len Min	순방향 패킷 최소 길이
Fwd Pkt Len Mean	순방향 패킷 평균 길이
Fwd Pkt Len Std	순방향 패킷 길이 표준편차
Fwd Seg Size Avg	순방향 평균 Segment 크기
Fwd Seg Size Min	순방향 최소 Segment 크기
공격과 정상 트래픽 사이에서 패킷 크기의 차이와 변동성을 보는 그룹이야.

4. 역방향 패킷 크기
Bwd는 목적지 → 출발지 방향이다.
컬럼	의미
Bwd Pkt Len Max	역방향 패킷 최대 길이
Bwd Pkt Len Min	역방향 패킷 최소 길이
Bwd Pkt Len Mean	역방향 패킷 평균 길이
Bwd Pkt Len Std	역방향 패킷 길이 표준편차
Bwd Seg Size Avg	역방향 평균 Segment 크기
여기서는 특히 요청과 응답의 비대칭성을 분석해볼 수 있어.
예를 들어:
Fwd는 많은데 Bwd는 적은 Flow Bwd가 Fwd보다 훨씬 큰 Flow
가 공격 클래스별로 다르게 나타나는지 확인하는 거야.

5. 전체 패킷 크기 통계
방향을 구분하지 않고 Flow 전체 패킷의 크기를 나타낸다.
컬럼	의미
Pkt Len Min	최소 패킷 길이
Pkt Len Max	최대 패킷 길이
Pkt Len Mean	평균 패킷 길이
Pkt Len Std	패킷 길이 표준편차
Pkt Len Var	패킷 길이 분산
Pkt Size Avg	평균 패킷 크기
여기서는 Mean만 보는 것보다 Std, Var도 중요해.
Mean → 일반적인 패킷 크기 Std → 크기가 얼마나 흔들리는가 Var → 변동 정도

6. IAT — 패킷 간 시간 간격
이 그룹은 이번 프로젝트에서 꽤 중요해.
IAT = Inter-Arrival Time, 즉 패킷과 다음 패킷 사이의 시간 간격이야.
전체 Flow
컬럼	의미
Flow IAT Mean	평균 패킷 도착 간격
Flow IAT Std	도착 간격 표준편차
Flow IAT Max	최대 도착 간격
Flow IAT Min	최소 도착 간격
Forward
컬럼	의미
Fwd IAT Tot	순방향 IAT 총합
Fwd IAT Mean	순방향 평균 IAT
Fwd IAT Std	순방향 IAT 표준편차
Fwd IAT Max	순방향 최대 IAT
Fwd IAT Min	순방향 최소 IAT
Backward
컬럼	의미

IAT는 결국 통신의 시간적 패턴을 표현한다고 이해하면 된다.
우리의 DNS/Beaconing AI로 확장할 때도 이런 시간 간격 개념이 중요해질 거야.

7. TCP Flag
TCP 연결의 상태와 제어 행동을 나타낸다.
컬럼	의미
FIN Flag Cnt	FIN Flag 개수
SYN Flag Cnt	SYN Flag 개수
RST Flag Cnt	RST Flag 개수
PSH Flag Cnt	PSH Flag 개수
ACK Flag Cnt	ACK Flag 개수
URG Flag Cnt	URG Flag 개수
ECE Flag Cnt	ECE Flag 개수
CWE Flag Count	CWR 관련 Flag count
Fwd PSH Flags	순방향 PSH
Bwd PSH Flags	역방향 PSH
Fwd URG Flags	순방향 URG
Bwd URG Flags	역방향 URG
대표적으로:
SYN ↓ 연결 시작 ACK ↓ 응답/확인 FIN ↓ 정상 연결 종료 RST ↓ 연결 강제 종료
정도로 먼저 이해하면 충분해.
이후 EDA에서 Label × SYN Flag Cnt, Label × RST Flag Cnt처럼 비교해서 실제 공격별 차이가 있는지 확인하면 된다.

8. Header / TCP Window
컬럼	의미
Fwd Header Len	순방향 Header 전체 길이
Bwd Header Len	역방향 Header 전체 길이
Init Fwd Win Byts	순방향 초기 TCP Window 크기
Init Bwd Win Byts	역방향 초기 TCP Window 크기
Fwd Act Data Pkts	실제 데이터를 포함하는 순방향 패킷 수
TCP 연결 특성이나 패킷 구조의 차이를 나타내는 Feature들이야.

9. Subflow
하나의 Flow 내부 통신을 세부적으로 표현한 통계야.
컬럼	의미
Subflow Fwd Pkts	Subflow 순방향 패킷 수
Subflow Fwd Byts	Subflow 순방향 Byte
Subflow Bwd Pkts	Subflow 역방향 패킷 수
Subflow Bwd Byts	Subflow 역방향 Byte
EDA 단계에서는 기존 Tot Fwd Pkts, TotLen Fwd Pkts등과 상관관계가 지나치게 높은지도 확인할 필요가 있어.
이게 Feature Reduction으로 연결될 수 있어.

10. Bulk Transfer
이름에 /b Avg가 들어간 부분이다.
컬럼	의미
Fwd Byts/b Avg	순방향 Bulk당 평균 Byte
Fwd Pkts/b Avg	순방향 Bulk당 평균 Packet
Fwd Blk Rate Avg	순방향 평균 Bulk 전송률
Bwd Byts/b Avg	역방향 Bulk당 평균 Byte

쉽게 생각하면 데이터가 묶음 단위로 대량 전송되는 특성을 표현하는 Feature 그룹이야.

11. Active / Idle
Flow의 활동 시간과 비활동 시간이다.
Active
컬럼	의미
Active Mean	평균 활성시간
Active Std	활성시간 표준편차
Active Max	최대 활성시간
Active Min	최소 활성시간
Idle
컬럼	의미
Idle Mean	평균 유휴시간
Idle Std	유휴시간 표준편차
Idle Max	최대 유휴시간
Idle Min	최소 유휴시간